<a href="https://colab.research.google.com/github/bhavyasripothunuri-max/Weather_and_tour_planner/blob/main/weather_and_tour_planner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Install Libraries
!pip -q install groq duckduckgo-search gradio requests

from groq import Groq
from duckduckgo_search import DDGS
import requests
import gradio as gr

# =========================
# Enter your Groq API Key
# =========================
client = Groq(api_key="YOUR_GROQ_API_KEY")

MODEL = "llama-3.3-70b-versatile"

# =========================
# Weather Function
# =========================
def weather(city):
    try:
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={
                "name": city,
                "count": 1
            },
            timeout=10
        ).json()

        if "results" not in geo:
            return f"Weather: City '{city}' not found."

        place = geo["results"][0]

        data = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": place["latitude"],
                "longitude": place["longitude"],
                "current": "temperature_2m"
            },
            timeout=10
        ).json()

        temp = data["current"]["temperature_2m"]

        return f"Current temperature in {city}: {temp}°C"

    except Exception as e:
        return f"Weather Error: {e}"


# =========================
# Search Function
# =========================
def search(query):
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=5))

        if not results:
            return "No search results found."

        output = ""

        for r in results:
            output += f"- {r['title']}\n"

        return output

    except Exception as e:
        return f"Search Error: {e}"


# =========================
# AI Agent
# =========================
def agent(question):

    # Decide which tool to use
    decision = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": """
Reply using ONLY one word:

weather
search
both
none
"""
            },
            {
                "role": "user",
                "content": question
            }
        ],
        temperature=0
    ).choices[0].message.content.strip().lower()

    info = ""

    # Weather Tool
    if decision in ["weather", "both"]:

        city = client.chat.completions.create(
            model=MODEL,
            messages=[
                {
                    "role": "system",
                    "content": "Return ONLY the city name."
                },
                {
                    "role": "user",
                    "content": question
                }
            ],
            temperature=0
        ).choices[0].message.content.strip()

        info += weather(city) + "\n\n"

    # Search Tool
    if decision in ["search", "both"]:
        info += search(question)

    # Final Response
    answer = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": """
You are a helpful travel assistant.

Answer the user's question using the information below.
If weather information is available, use it.
If search results are available, use them.
Give a natural and friendly response.
"""
            },
            {
                "role": "user",
                "content": f"""
Question:
{question}

Information:
{info}
"""
            }
        ]
    )

    return answer.choices[0].message.content


# =========================
# Test
# =========================
print(agent("Should I carry an umbrella for Goa and suggest places to visit?"))


# =========================
# Gradio App
# =========================
demo = gr.ChatInterface(
    fn=lambda message, history: agent(message),
    title="🌦️ Weather + Trip Planner",
    description="Ask about weather, travel places, or both."
)

demo.launch(share=True)

AuthenticationError: Error code: 401 - {'error': {'message': 'Invalid API Key', 'type': 'invalid_request_error', 'code': 'invalid_api_key'}}